# Práctica obligatoria · Del problema de churn a una primera predicción

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)  
**Carácter:** obligatoria; no requiere entrega ni calificación.

> Realizá solamente uno de los dos recorridos equivalentes: **SQL + Python** o **SQL + R**.

Esta práctica funciona como **andamiaje inmediato para la Entrega 5 del TPI**. El objetivo no es evaluar formalmente el modelo, sino recorrer de manera guiada el procedimiento mínimo necesario para pasar de un problema de clasificación a una primera predicción.

El notebook es autocontenido: genera internamente un conjunto de datos de churn y no requiere archivos auxiliares.


## 1. Preparar el entorno

Ejecutá la celda siguiente. No necesitás descargar ningún dataset ni archivo adicional.


In [ ]:
#@title Preparación automática del entorno y de los datos (ejecutar sin modificar)
import importlib.util, subprocess, sys, sqlite3
import numpy as np
import pandas as pd

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(42)
n = 600

tenure = rng.integers(1, 73, n)
contract = rng.choice(
    ["Month-to-month", "One year", "Two year"],
    size=n,
    p=[0.56, 0.25, 0.19]
)
tech_support = rng.choice(["Yes", "No"], size=n, p=[0.42, 0.58])
monthly = np.clip(rng.normal(70, 22, n), 20, 125).round(2)
total = np.clip(monthly * tenure + rng.normal(0, 140, n), 20, None).round(2)

logit = (
    -1.4
    + 1.25 * (contract == "Month-to-month")
    + 0.75 * (tech_support == "No")
    + 0.018 * (monthly - 70)
    - 0.032 * (tenure - 24)
)
prob = 1 / (1 + np.exp(-logit))
churn = np.where(rng.random(n) < prob, "Yes", "No")

clientes = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(1, n + 1)],
    "tenure": tenure,
    "Contract": contract,
    "TechSupport": tech_support,
    "MonthlyCharges": monthly,
    "TotalCharges": total,
    "Churn": churn
})

con = sqlite3.connect(":memory:")
clientes.to_sql("churn", con, index=False, if_exists="replace")

print(f"Entorno preparado: {len(clientes)} clientes y {clientes.shape[1]} variables.")


## 2. Seleccionar con SQL las variables que usará el modelo

La variable objetivo es `Churn`. Vamos a trabajar con un conjunto pequeño de predictores para mantener el foco en el procedimiento.


In [ ]:
consulta = """
SELECT
    tenure,
    Contract,
    TechSupport,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM churn
"""
df = pd.read_sql_query(consulta, con)
df.head()


## 3. Preparar X e y

- `X` contiene las variables predictoras.
- `y` contiene la variable objetivo.
- Las variables categóricas se convierten a formato numérico antes de entrenar.


In [ ]:
y = (df["Churn"] == "Yes").astype(int)

X = pd.get_dummies(
    df.drop(columns="Churn"),
    columns=["Contract", "TechSupport"],
    drop_first=False
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## 4. Entrenar un modelo sencillo

Vamos a usar un árbol de decisión pequeño. No buscamos el mejor modelo posible, sino un modelo fácil de ejecutar e interpretar.


In [ ]:
modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

predicciones = modelo.predict(X_test)

print("Modelo entrenado.")


## 5. Observar algunas predicciones

Compará el resultado real con la predicción del modelo en algunos casos.


In [ ]:
resultado = pd.DataFrame({
    "real": y_test.to_numpy(),
    "prediccion": predicciones
}).head(12)

resultado


## 6. Interpretar antes de evaluar

Respondé brevemente dentro del notebook:

1. ¿Qué representa `y` en este problema?
2. ¿Por qué fue necesario transformar las variables categóricas antes de entrenar?
3. Elegí uno de los casos mostrados. Si el valor real y la predicción no coinciden, ¿qué significa ese desacuerdo?

### Cierre

Ya recorriste el patrón mínimo que vas a transferir al **TPI · Entrega 5**:

**seleccionar datos → preparar X e y → dividir entrenamiento/prueba → entrenar → predecir → interpretar.**

Todavía no corresponde evaluar formalmente el modelo con métricas. Ese trabajo se desarrollará en la **Unidad N° 6**.


In [ ]:
con.close()